# Sensex — The Last-Two-Session Move Into Expiry, Bucketed By India VIX

One number per expiry: the move from the **open of the session before expiry** to the **close of the expiry session** —
`Thursday open to Friday close until Dec 2024, Monday open to Tuesday close through Aug 2025, Wednesday open to Thursday close after it` — and what India VIX was quoting when that position was opened.

The table this notebook exists for is the first one below: a VIX bucket on the left, the average move that followed on
the right. It answers the practical question a short-premium trader asks on the day before expiry — *given VIX is 12.4
right now, how far does this index typically travel between here and settlement?*

Two averages are reported side by side, and the difference between them matters:

- **Avg |move|** — the mean of the *absolute* move. This is the "how far does it travel" number, the one that should be
  compared against the premium on offer. It is the headline column.
- **Avg move (signed)** — the mean of the signed move. Ups cancel downs, so this sits near zero in every bucket and says
  only whether the window carried a directional drift. A signed average near 0.0% with an absolute average of 0.6% is
  the normal picture, not a contradiction.

VIX is annualised, so it is de-annualised for comparison: the quote implies a one-session σ of `VIX / √252` and a
`WINDOW_SESSIONS`-session σ of `VIX / √252 × √WINDOW_SESSIONS`. For a zero-mean normal the *expected absolute* move is
`σ × √(2/π) ≈ 0.80σ`, so that — not σ itself — is the column the realised `Avg |move|` is held against.

Source data:

- `data/raw/option_selling/sensex/sensex_expiry_dates.csv` — scheduled and actual expiry dates (monthly-only before the
  May 2023 weekly launch; those cycles are dropped, see `MAX_EXPIRY_GAP_DAYS`)
- `data/raw/option_selling/sensex/sensex_hourly_2020-01-01_2026-11-06.json` — hourly OHLC candles (data starts 2022-01-03)
- `data/raw/option_selling/india_vix_hourly_2020-01-01_2026-11-06.json` — hourly India VIX (data starts 2022-01-03)

One caveat specific to this notebook: **India VIX is computed from Nifty 50 options, not Sensex options.** There is no
published Sensex VIX, so the bucketing here uses Nifty's implied vol as a proxy for the volatility regime. The two
indices are closely correlated, but the proxy is an assumption, and a Sensex-specific move priced off a Nifty quote is
the thing to keep in mind when reading the implied columns.

The companion notebook for the other index is `notebooks/option_selling/nifty50/expiry_two_session_move_by_vix.ipynb` and runs the identical code path; only the parameters cell
differs.

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    """Walk up from the notebook's location until the folder holding data/raw is found."""
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "option_selling"

# --- Which index this notebook measures ------------------------------------
INDEX_NAME = "SENSEX"
INDEX_JSON = RAW_DIR / "sensex" / "sensex_hourly_2020-01-01_2026-11-06.json"
EXPIRY_CSV = RAW_DIR / "sensex" / "sensex_expiry_dates.csv"
# ---------------------------------------------------------------------------

VIX_JSON = RAW_DIR / "india_vix_hourly_2020-01-01_2026-11-06.json"
VIX_NAME = "INDIA VIX"

# --- Parameters -------------------------------------------------------------
ENTRY_SESSIONS_BEFORE = 1                             # entry session = N regular sessions before expiry (1 = expiry_day - 1)
ENTRY_TIME = "09:15"                                  # entry candle start (IST); its open is the entry price
VIX_ANCHOR = "entry_open"                             # "entry_open" (VIX at entry) | "prev_close" (last VIX close before entry)
VIX_BANDS = [0, 10, 11, 12, 13, 14, 15, 16, 18, 100]  # VIX bucket edges, in annualised vol points
TOUCH_DISTANCES = [0.5, 1.0, 1.5, 2.0]                # % from the entry price, for the touch table
MAX_EXPIRY_GAP_DAYS = 12                              # longer expiry-to-expiry gaps are monthly-only cycles and are dropped
TRADING_DAYS = 252                                    # sessions per year, used to de-annualise VIX
REGULAR_SESSION_END_HOUR = 16                         # candles stamped later are Muhurat/special sessions
REGULAR_SESSION_MIN_CANDLES = 6                       # weekday sessions with fewer candles are not normal trading days
# ---------------------------------------------------------------------------

WINDOW_SESSIONS = ENTRY_SESSIONS_BEFORE + 1           # sessions the move spans: the entry session(s) plus the expiry session
HOLD_LABEL = f"open on expiry_day-{ENTRY_SESSIONS_BEFORE} to close on expiry_day"   # prose and column headers
HOLD_TITLE = f"Open On expiry_day−{ENTRY_SESSIONS_BEFORE} → Close On expiry_day"      # figure titles
ANNUALISE = math.sqrt(TRADING_DAYS)
HALF_NORMAL = math.sqrt(2 / math.pi)                  # E|X| / sigma for a zero-mean normal: turns a sigma into an expected |move|


def rank_corr(left: pd.Series, right: pd.Series) -> float:
    """Spearman rank correlation as Pearson on average ranks - avoids a SciPy dependency."""
    paired = pd.concat([left, right], axis=1).dropna()
    return paired.iloc[:, 0].rank().corr(paired.iloc[:, 1].rank())


def band_labels(edges: list[float]) -> list[str]:
    """Readable labels for pd.cut buckets: open-ended on both tails, `lo-hi` in between."""
    labels = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        if lo == edges[0]:
            labels.append(f"≤ {hi:g}")
        elif hi == edges[-1]:
            labels.append(f"> {lo:g}")
        else:
            labels.append(f"{lo:g}–{hi:g}")
    return labels


BAND_LABELS = band_labels(VIX_BANDS)

# Fixed categorical hues (slots 1-3), never cycled. Diverging poles with a neutral midpoint: up = warm, down = cool.
INDEX_HEX, UP_HEX, VIX_HEX = "#2a78d6", "#eb6834", "#1baf7a"
UP_COLOR, DOWN_COLOR, NEUTRAL = (235, 104, 52), (42, 120, 214), (240, 239, 236)
SEQUENTIAL = [[0, "#cde2fb"], [0.5, "#3987e5"], [1, "#0d366b"]]   # one hue, light to dark
GRID, MUTED_INK, HEADER_FILL = "#eceae6", "#8a8880", "#2b2d42"

pd.options.display.float_format = "{:,.2f}".format
print(f"{INDEX_NAME}: {HOLD_LABEL} ({WINDOW_SESSIONS} sessions), VIX anchored on {VIX_ANCHOR}")

SENSEX: open on expiry_day-1 to close on expiry_day (2 sessions), VIX anchored on entry_open


## Load The Index And VIX Onto One Session Calendar

Both dumps share the same hourly grid, so the clean-up is identical for each:

- **Special sessions are dropped.** A regular session is a **weekday** with at least `REGULAR_SESSION_MIN_CANDLES`
  candles stamped before `REGULAR_SESSION_END_HOUR`. The weekday test removes full weekend sessions (Budget Saturdays,
  exchange DR drills, which carry a complete seven-candle grid and so cannot be caught by a candle count); the candle
  count removes Muhurat stubs. Left in, a one-candle session would register as an enormous or a zero move.
- **The calendar is intersected across both series**, so every cycle below has an index print *and* a VIX print on each
  of its sessions. No bucket is formed from a VIX level that has to be carried forward.
- **Hourly candles are kept** as well as collapsed into one OHLC row per session. The daily rows give the open and the
  close that define the move; the hourly rows give the intra-window high and low that the touch section needs.
- Both files are stamped `2020-01-01` in their names but start **2022-01-03**; the printed range is the real one.

VIX is loaded through the same OHLC path, but it is a *level*, not a price — only its open and close are read, and never
differenced as a return.

In [2]:
def load_hourly(path: Path) -> pd.DataFrame:
    """Hourly OHLC from a candle dump, de-duplicated and stamped in IST-naive time."""
    with open(path) as f:
        raw = json.load(f)
    frame = pd.DataFrame(
        raw["data"]["candles"],
        columns=["timestamp", "open", "high", "low", "close", "volume", "oi"],
    )
    frame["timestamp"] = (
        pd.to_datetime(frame["timestamp"], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
    )
    frame = frame.drop_duplicates("timestamp").set_index("timestamp").sort_index()
    return frame[["open", "high", "low", "close"]].apply(pd.to_numeric)


def regular_sessions(hourly_frame: pd.DataFrame) -> pd.DatetimeIndex:
    """Weekday sessions carrying a full hourly grid - Muhurat and weekend special sessions excluded."""
    intraday = hourly_frame[hourly_frame.index.hour < REGULAR_SESSION_END_HOUR]
    per_session = intraday.groupby(intraday.index.normalize()).size()
    return per_session.index[
        (per_session.index.dayofweek < 5) & (per_session.to_numpy() >= REGULAR_SESSION_MIN_CANDLES)
    ]


def session_candles(hourly_frame: pd.DataFrame, keep: pd.DatetimeIndex) -> pd.DataFrame:
    """Hourly candles of the kept sessions, tagged with the session they belong to."""
    intraday = hourly_frame[hourly_frame.index.hour < REGULAR_SESSION_END_HOUR].copy()
    intraday["session"] = intraday.index.normalize()
    return intraday[intraday["session"].isin(keep)]


def to_daily(hourly_frame: pd.DataFrame, keep: pd.DatetimeIndex) -> pd.DataFrame:
    """Collapse hourly candles into one OHLC row per regular session."""
    grouped = session_candles(hourly_frame, keep).groupby("session")
    return pd.DataFrame(
        {
            "open": grouped["open"].first(),
            "high": grouped["high"].max(),
            "low": grouped["low"].min(),
            "close": grouped["close"].last(),
        }
    )


raw_hourly = {INDEX_NAME: load_hourly(INDEX_JSON), VIX_NAME: load_hourly(VIX_JSON)}

sessions = regular_sessions(raw_hourly[INDEX_NAME])
for frame in raw_hourly.values():
    sessions = sessions.intersection(regular_sessions(frame))

price = session_candles(raw_hourly[INDEX_NAME], sessions)       # hourly candles, tagged with their session
vix_candles = session_candles(raw_hourly[VIX_NAME], sessions)
price_daily = to_daily(raw_hourly[INDEX_NAME], sessions)
vix_daily = to_daily(raw_hourly[VIX_NAME], sessions)
vix_close = vix_daily["close"]

for name, frame in raw_hourly.items():
    dropped = sorted(set(frame.index.normalize().unique()) - set(sessions))
    print(
        f"{name:<10} {len(frame):>6,} hourly candles -> {len(to_daily(frame, sessions)):>5,} regular sessions, "
        f"{len(dropped)} session(s) dropped: " + ", ".join(f"{d:%Y-%m-%d %a}" for d in dropped)
    )

print(
    f"\nCommon calendar: {len(sessions):,} sessions, {sessions.min():%Y-%m-%d} to {sessions.max():%Y-%m-%d} "
    f"({(sessions.max() - sessions.min()).days / 365.25:.2f} years)"
)
print(f"Candles per kept session: {sorted(price.groupby('session').size().unique())}")
print(f"{INDEX_NAME} close: {price_daily['close'].min():,.2f} to {price_daily['close'].max():,.2f}")
print(
    f"{VIX_NAME} close: min {vix_close.min():.2f}, median {vix_close.median():.2f}, "
    f"mean {vix_close.mean():.2f}, max {vix_close.max():.2f}"
)

SENSEX      8,383 hourly candles -> 1,193 regular sessions, 10 session(s) dropped: 2022-08-09 Tue, 2022-10-24 Mon, 2023-11-12 Sun, 2024-01-20 Sat, 2024-03-02 Sat, 2024-05-18 Sat, 2024-11-01 Fri, 2025-02-01 Sat, 2025-10-21 Tue, 2026-02-01 Sun
INDIA VIX   8,382 hourly candles -> 1,193 regular sessions, 9 session(s) dropped: 2022-10-24 Mon, 2023-11-12 Sun, 2024-01-20 Sat, 2024-03-02 Sat, 2024-05-18 Sat, 2024-11-01 Fri, 2025-02-01 Sat, 2025-10-21 Tue, 2026-02-01 Sun

Common calendar: 1,193 sessions, 2022-01-03 to 2026-11-06 (4.84 years)
Candles per kept session: [np.int64(7)]
SENSEX close: 51,478.71 to 85,810.29
INDIA VIX close: min 9.13, median 13.95, mean 14.95, max 32.75


## Build The Expiry Chain

Expiry dates come from the exchange calendar; the chain is cleaned the same way as in the other option-selling
notebooks in this folder, so the cycle counts line up with them:

- **One expiry per week.** When a Mon-Sun week holds two expiry dates (an off-cycle monthly landing two days before that
  week's weekly, as the Sept-Dec 2025 calendar does) only the later one is kept.
- **Weekly cadence only.** Cadence is measured on the *full* calendar, before it is clipped to the candle range, so the
  first expiry inside the data is judged against its real predecessor. Expiries more than `MAX_EXPIRY_GAP_DAYS` calendar
  days after the previous one are monthly-only cycles — every Sensex expiry before the May 2023 weekly launch — and are
  dropped. Set `MAX_EXPIRY_GAP_DAYS` large to put them back.
- **Each expiry is snapped back** to the last regular session on or before it, so a holiday-shifted expiry still settles
  on a real closing print. Duplicates after snapping are removed.

In [3]:
expiry_df = pd.read_csv(EXPIRY_CSV, parse_dates=["Actual_Expiry_Date"])
calendar = expiry_df["Actual_Expiry_Date"].dt.normalize().drop_duplicates().sort_values().reset_index(drop=True)

earlier_in_week = calendar.dt.to_period("W").duplicated(keep="last")   # off-cycle expiry sharing a Mon-Sun week
off_cycle = calendar[earlier_in_week]
calendar = calendar[~earlier_in_week].reset_index(drop=True)

cadence_days = calendar.diff().dt.days
weekly = calendar[cadence_days <= MAX_EXPIRY_GAP_DAYS]                 # cadence judged on the full calendar

in_range = weekly[(weekly >= sessions.min()) & (weekly <= sessions.max())]
monthly_only = calendar[(cadence_days > MAX_EXPIRY_GAP_DAYS) & (calendar >= sessions.min()) & (calendar <= sessions.max())]
snapped = pd.Series(sessions[sessions.searchsorted(in_range, side="right") - 1])
expiry_sessions = snapped.drop_duplicates().reset_index(drop=True)
shifted_count = int((snapped.to_numpy() != in_range.to_numpy()).sum())

print(f"Calendar rows: {len(expiry_df)}, {len(off_cycle)} dropped as an earlier second expiry in the same week: "
      + ", ".join(f"{d:%Y-%m-%d %a}" for d in off_cycle))
print(f"{len(monthly_only)} in-range expiry(ies) dropped as monthly-only (gap > {MAX_EXPIRY_GAP_DAYS} days)"
      + (": " + ", ".join(f"{d:%Y-%m-%d}" for d in monthly_only) if len(monthly_only) else ""))
print(f"{len(expiry_sessions)} weekly expiries in range, {shifted_count} snapped back to the previous regular session")
print(f"Expiry weekday: {expiry_sessions.dt.day_name().value_counts().to_dict()}")
print(f"First {expiry_sessions.iloc[0]:%Y-%m-%d}, last {expiry_sessions.iloc[-1]:%Y-%m-%d}")

Calendar rows: 218, 4 dropped as an earlier second expiry in the same week: 2025-09-30 Tue, 2025-10-28 Tue, 2025-11-25 Tue, 2025-12-30 Tue
17 in-range expiry(ies) dropped as monthly-only (gap > 12 days): 2022-01-27, 2022-02-24, 2022-03-31, 2022-04-28, 2022-05-26, 2022-06-30, 2022-07-28, 2022-08-25, 2022-09-29, 2022-10-27, 2022-11-24, 2022-12-29, 2023-01-25, 2023-02-23, 2023-03-29, 2023-04-27, 2023-05-19
173 weekly expiries in range, 1 snapped back to the previous regular session
Expiry weekday: {'Friday': 79, 'Thursday': 55, 'Tuesday': 34, 'Wednesday': 5}
First 2023-05-26, last 2026-09-17


## Measure Every Expiry Window

For each expiry, the **entry** is the first hourly candle at or after `ENTRY_TIME` on the regular session
`ENTRY_SESSIONS_BEFORE` before it — the session-open candle by default — and the **exit** is the close of the expiry
session. Holidays are handled by counting regular sessions, so `expiry_day - 1` is always the session that actually
traded before expiry, never a calendar day that did not.

Per cycle:

| Column | Definition |
|---|---|
| `entry_price` | open of the entry candle; everything below is measured against it |
| `move_pct` | **the headline move** — entry price to the close of the expiry session |
| `abs_move_pct` | its magnitude, which is what the average in the main table reports |
| `max_up_pct` / `max_down_pct` | best and worst hourly print inside the window, vs `entry_price` |
| `entry_day_pct` | entry price to the close of the entry session |
| `gap_pct` | entry session close to the expiry session open — the overnight no position can manage |
| `expiry_day_pct` | expiry session open to its close |
| `vix_entry_open` | VIX at the entry candle — known at the moment the position is opened |
| `vix_prev_close` | VIX close of the session *before* the entry, the fully-settled alternative anchor |
| `implied_sigma_pct` | `vix / √252 × √WINDOW_SESSIONS` — what the quote implies for this window |
| `implied_abs_move_pct` | `implied_sigma_pct × √(2/π)` — the expected *absolute* move under a normal |

`vix` is whichever anchor `VIX_ANCHOR` selects, and `band` is its `VIX_BANDS` bucket. The legs (`entry_day_pct`,
`gap_pct`, `expiry_day_pct`) compound to `move_pct` and are carried so the window can be taken apart later.

In [4]:
def first_candle_at(frame: pd.DataFrame, session: pd.Timestamp, time_str: str) -> pd.Timestamp | None:
    """Timestamp of the first candle at or after `time_str` on `session`, or None if the session has none."""
    day = frame[frame["session"] == session]
    at_or_after = day.index[day.index.strftime("%H:%M") >= time_str]
    return at_or_after[0] if len(at_or_after) else None


def build_cycles(expiries: pd.Series) -> tuple[pd.DataFrame, list[pd.Timestamp]]:
    """One row per expiry: the entry-to-expiry-close move and the VIX level that was quoted at entry."""
    rows, skipped = [], []
    for expiry in expiries:
        pos = sessions.get_loc(expiry)
        if pos - ENTRY_SESSIONS_BEFORE < 1:          # need the entry session and the session before it (prev-close anchor)
            skipped.append(expiry)
            continue
        entry_session = sessions[pos - ENTRY_SESSIONS_BEFORE]
        entry_ts = first_candle_at(price, entry_session, ENTRY_TIME)
        vix_ts = first_candle_at(vix_candles, entry_session, ENTRY_TIME)
        if entry_ts is None or vix_ts is None:
            skipped.append(expiry)
            continue

        window = price[(price.index >= entry_ts) & (price["session"] <= expiry)]
        entry_price = window["open"].iloc[0]
        expiry_close = window["close"].iloc[-1]
        entry_day, expiry_day = price_daily.loc[entry_session], price_daily.loc[expiry]
        vix_at_entry = vix_candles.at[vix_ts, "open"]
        vix_prev = vix_close.loc[sessions[pos - ENTRY_SESSIONS_BEFORE - 1]]

        rows.append(
            {
                "expiry": expiry,
                "expiry_day": expiry.day_name(),
                "entry_session": entry_session,
                "entry_ts": entry_ts,
                "entry_price": entry_price,
                "expiry_close": expiry_close,
                "move_pct": (expiry_close - entry_price) / entry_price * 100,
                "max_up_pct": (window["high"].max() - entry_price) / entry_price * 100,
                "max_down_pct": (window["low"].min() - entry_price) / entry_price * 100,
                "entry_day_pct": (entry_day["close"] - entry_price) / entry_price * 100,
                "gap_pct": (expiry_day["open"] - entry_day["close"]) / entry_day["close"] * 100,
                "expiry_day_pct": (expiry_day["close"] - expiry_day["open"]) / expiry_day["open"] * 100,
                "vix_entry_open": vix_at_entry,
                "vix_prev_close": vix_prev,
                "vix_expiry_close": vix_close.loc[expiry],
                "sessions_in_window": window["session"].nunique(),
                "candles": len(window),
            }
        )
    return pd.DataFrame(rows), skipped


cycles, skipped_expiries = build_cycles(expiry_sessions)
cycles["abs_move_pct"] = cycles["move_pct"].abs()
cycles["vix"] = cycles["vix_entry_open"] if VIX_ANCHOR == "entry_open" else cycles["vix_prev_close"]
cycles["band"] = pd.cut(cycles["vix"], VIX_BANDS, labels=BAND_LABELS)
cycles["implied_sigma_pct"] = cycles["vix"] / ANNUALISE * math.sqrt(WINDOW_SESSIONS)
cycles["implied_abs_move_pct"] = cycles["implied_sigma_pct"] * HALF_NORMAL
cycles["year"] = cycles["expiry"].dt.year

compounded = (1 + cycles["entry_day_pct"] / 100) * (1 + cycles["gap_pct"] / 100) * (1 + cycles["expiry_day_pct"] / 100)
assert np.allclose(compounded * 100 - 100, cycles["move_pct"]), "the three legs must compound to the headline move"

late = cycles[cycles["entry_ts"].dt.strftime("%H:%M") != ENTRY_TIME]
print(f"{len(cycles)} expiry cycles measured, {HOLD_LABEL}")
print(f"Entry weekday: {cycles['entry_session'].dt.day_name().value_counts().to_dict()}")
print(f"Sessions per window: {cycles['sessions_in_window'].value_counts().to_dict()}, "
      f"hourly candles per window: {cycles['candles'].median():.0f} (median)")
print(f"{len(late)} cycle(s) had no {ENTRY_TIME} candle and entered on that session's next candle")
if skipped_expiries:
    print(f"{len(skipped_expiries)} expiry(ies) skipped for want of enough prior sessions: "
          + ", ".join(f"{d:%Y-%m-%d}" for d in skipped_expiries))
print(f"VIX at entry: min {cycles['vix'].min():.2f}, median {cycles['vix'].median():.2f}, max {cycles['vix'].max():.2f}")
display(cycles.tail(8)[["expiry", "entry_session", "entry_price", "expiry_close", "move_pct", "max_up_pct",
                        "max_down_pct", "vix", "band"]])

173 expiry cycles measured, open on expiry_day-1 to close on expiry_day
Entry weekday: {'Thursday': 76, 'Wednesday': 56, 'Monday': 33, 'Tuesday': 6, 'Friday': 2}
Sessions per window: {2: 173}, hourly candles per window: 14 (median)
0 cycle(s) had no 09:15 candle and entered on that session's next candle
VIX at entry: min 9.68, median 13.27, max 27.89


,expiry,entry_session,entry_price,expiry_close,move_pct,max_up_pct,max_down_pct,vix,band
165,2026-07-30,2026-07-29,"77,423.77","77,835.43",0.53,0.75,-0.12,12.56,12–13
166,2026-08-06,2026-08-05,"79,055.38","78,954.76",-0.13,0.00,-0.97,12.19,12–13
167,2026-08-13,2026-08-12,"78,263.33","78,079.96",-0.23,0.00,-0.98,11.86,11–12
168,2026-08-20,2026-08-19,"77,218.05","77,537.72",0.41,0.51,-0.51,11.39,11–12
169,2026-08-27,2026-08-26,"77,892.10","76,933.59",-1.23,0.12,-1.23,11.08,11–12
170,2026-09-03,2026-09-02,"76,471.32","76,152.86",-0.42,0.59,-0.44,11.49,11–12
171,2026-09-10,2026-09-09,"75,216.22","74,902.59",-0.42,0.05,-0.82,11.23,11–12
172,2026-09-17,2026-09-16,"74,249.31","74,314.59",0.09,0.58,-0.36,13.43,13–14


## The Table — Average Move By VIX Bucket

The requested table first, in the plainest form: the VIX bucket at entry, how many expiry cycles landed in it, and the
average move that followed. Buckets are left-open and right-closed — `12–13` means `12 < VIX ≤ 13` — and buckets
holding no cycles are left out and listed underneath.

Read `Avg |move|` as *typical distance travelled*, which is the number that matters against a premium. `Avg move
(signed)` is printed beside it only to show that the drift is not what drives the first column.

In [5]:
present = [label for label in BAND_LABELS if (cycles["band"] == label).sum() > 0]
empty = [label for label in BAND_LABELS if label not in present]
grouped = {label: cycles[cycles["band"] == label] for label in present}

bands = pd.DataFrame(
    [
        {
            "VIX band": label,
            "Cycles": len(bucket),
            "Share %": len(bucket) / len(cycles) * 100,
            "Mean VIX": bucket["vix"].mean(),
            "Avg |move| %": bucket["abs_move_pct"].mean(),
            "Median |move| %": bucket["abs_move_pct"].median(),
            "Avg move %": bucket["move_pct"].mean(),
            "Move sigma %": bucket["move_pct"].std(),
            "Closed up %": (bucket["move_pct"] > 0).mean() * 100,
            "Best %": bucket["move_pct"].max(),
            "Worst %": bucket["move_pct"].min(),
            "Implied sigma %": bucket["implied_sigma_pct"].mean(),
            "Implied E|move| %": bucket["implied_abs_move_pct"].mean(),
        }
        for label, bucket in grouped.items()
    ]
)
bands["Actual / implied"] = bands["Avg |move| %"] / bands["Implied E|move| %"]

headline = "\n".join(
    [f"| vix | cycles | avg move ({HOLD_LABEL}) |", "| --- | --- | --- |"]
    + [f"| {row['VIX band']} | {row['Cycles']:.0f} | {row['Avg |move| %']:.2f}% |" for _, row in bands.iterrows()]
    + [f"| **all** | **{len(cycles)}** | **{cycles['abs_move_pct'].mean():.2f}%** |"]
)
display(Markdown(f"**{INDEX_NAME} — average absolute move, {HOLD_LABEL}**\n\n" + headline))
print(f"Buckets with no cycles: {', '.join(empty) if empty else 'none'}")

**SENSEX — average absolute move, open on expiry_day-1 to close on expiry_day**

| vix | cycles | avg move (open on expiry_day-1 to close on expiry_day) |
| --- | --- | --- |
| ≤ 10 | 2 | 0.40% |
| 10–11 | 14 | 0.61% |
| 11–12 | 41 | 0.58% |
| 12–13 | 24 | 0.75% |
| 13–14 | 26 | 0.96% |
| 14–15 | 24 | 0.70% |
| 15–16 | 12 | 0.66% |
| 16–18 | 14 | 0.99% |
| > 18 | 16 | 1.14% |
| **all** | **173** | **0.77%** |

Buckets with no cycles: none


In [6]:
def diverging_fill(values: pd.Series, pole_up=UP_COLOR, pole_down=DOWN_COLOR) -> list[str]:
    """Blend from the neutral surface toward the up/down pole in proportion to magnitude."""
    peak = values.abs().max() or 1.0
    fills = []
    for value in values:
        pole = pole_up if value >= 0 else pole_down
        mix = min(abs(value) / peak, 1.0)
        fills.append("rgb({},{},{})".format(*[int(NEUTRAL[i] + (pole[i] - NEUTRAL[i]) * mix) for i in range(3)]))
    return fills


def shade(values: pd.Series, color=UP_COLOR) -> list[str]:
    """Single-pole shading: neutral surface toward `color` in proportion to each value's magnitude."""
    magnitude = values.abs()
    peak = magnitude.max() or 1.0
    return [
        "rgb({},{},{})".format(*[int(NEUTRAL[i] + (color[i] - NEUTRAL[i]) * t) for i in range(3)])
        for t in (magnitude / peak).clip(0, 1)
    ]


white = ["white"] * len(bands)
fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[62, 55, 50, 55, 70, 70, 70, 58, 58, 55, 58, 68, 70, 62],
            header=dict(
                values=[
                    "VIX<br>band", "Cycles", "Share", "Mean<br>VIX", "<b>Avg |move|</b>", "Median<br>|move|",
                    "Avg move<br>(signed)", "Move<br>σ", "Closed<br>up", "Best", "Worst",
                    "Implied<br>σ", "Implied<br>E|move|", "Actual /<br>implied",
                ],
                fill_color=HEADER_FILL, font=dict(color="white", size=10), align="center", height=46,
            ),
            cells=dict(
                values=[
                    bands["VIX band"],
                    bands["Cycles"],
                    bands["Share %"].map("{:.0f}%".format),
                    bands["Mean VIX"].map("{:.1f}".format),
                    bands["Avg |move| %"].map("{:.2f}%".format),
                    bands["Median |move| %"].map("{:.2f}%".format),
                    bands["Avg move %"].map("{:+.2f}%".format),
                    bands["Move sigma %"].map("{:.2f}%".format),
                    bands["Closed up %"].map("{:.0f}%".format),
                    bands["Best %"].map("{:+.2f}%".format),
                    bands["Worst %"].map("{:+.2f}%".format),
                    bands["Implied sigma %"].map("{:.2f}%".format),
                    bands["Implied E|move| %"].map("{:.2f}%".format),
                    bands["Actual / implied"].map("{:.2f}x".format),
                ],
                fill_color=[
                    ["#f4f3f0"] * len(bands), white, white, white,
                    shade(bands["Avg |move| %"]), shade(bands["Median |move| %"]),
                    diverging_fill(bands["Avg move %"]), white, white,
                    shade(bands["Best %"], UP_COLOR), shade(bands["Worst %"], DOWN_COLOR),
                    white, white, diverging_fill(bands["Actual / implied"] - 1),
                ],
                align="center", height=27, font=dict(size=11),
            ),
        )
    ]
)
fig.update_layout(
    title=f"{INDEX_NAME} — Move {HOLD_TITLE}, By India VIX At Entry ({len(cycles)} expiry cycles)",
    height=140 + 27 * len(bands) + 60, margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

## Realised Against What VIX Was Charging

Same units on both bars — percent move over the same two sessions — so they belong on one axis. The left bar of each
pair is what actually happened on average; the right bar is what the VIX quote implied for the window, converted to an
expected absolute move (`σ × √(2/π)`). A left bar shorter than its right bar is the variance risk premium showing up in
this specific window; taller means the window delivered more travel than the quote priced.

Bars are labelled directly as well as coloured, and the cycle count sits under each bucket, because a bucket of six
expiries should not be read like a bucket of sixty.

In [7]:
axis_labels = [f"{row['VIX band']}<br><span style='font-size:10px;color:{MUTED_INK}'>n={row['Cycles']:.0f}</span>"
               for _, row in bands.iterrows()]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=axis_labels, y=bands["Avg |move| %"], name="Realised avg |move|",
        marker_color=INDEX_HEX, marker_line=dict(color="white", width=2),
        text=bands["Avg |move| %"].map("{:.2f}%".format), textposition="outside", textfont=dict(size=11),
        hovertemplate="VIX %{x}<br>realised avg |move| %{y:.2f}%<extra></extra>",
    )
)
fig.add_trace(
    go.Bar(
        x=axis_labels, y=bands["Implied E|move| %"], name="VIX-implied E|move|",
        marker_color=VIX_HEX, marker_line=dict(color="white", width=2),
        text=bands["Implied E|move| %"].map("{:.2f}%".format), textposition="outside", textfont=dict(size=11),
        hovertemplate="VIX %{x}<br>implied E|move| %{y:.2f}%<extra></extra>",
    )
)
fig.update_layout(
    title=f"{INDEX_NAME} — Realised Vs VIX-Implied Move, {HOLD_TITLE}",
    template="plotly_white", height=460, bargap=0.28, bargroupgap=0.04,
    legend=dict(orientation="h", y=1.08, x=0, title_text=""),
    margin=dict(t=90, b=60, l=60, r=20),
)
fig.update_xaxes(title_text=f"India VIX at entry ({VIX_ANCHOR})", gridcolor=GRID)
fig.update_yaxes(title_text="Move over the window (%)", gridcolor=GRID, rangemode="tozero")
fig.show()

## Dispersion Inside Each Bucket

An average hides the tail that actually decides a short option position, so the signed move of every cycle is shown as a
box per bucket: the box spans the interquartile range, the whiskers reach the extremes, and each point is one expiry.
Buckets are shaded light-to-dark by VIX level — one hue, because the encoding is magnitude, not identity. The zero rule
marks a flat settlement.

The Spearman correlation underneath is the honest summary of "does VIX predict the move": it is computed on every cycle,
not on the nine bucket averages, so it is not flattered by the bucketing.

In [8]:
shades = [SEQUENTIAL[0][1], "#9cc6f5", "#6ba6ee", "#3987e5", "#2a78d6", "#1f5fa8", "#174a85", "#0d366b", "#07213f"]

fig = go.Figure()
for position, (label, bucket) in enumerate(grouped.items()):
    fig.add_trace(
        go.Box(
            y=bucket["move_pct"], name=f"{label}<br><span style='font-size:10px;color:{MUTED_INK}'>n={len(bucket)}</span>",
            marker_color=shades[position % len(shades)], line=dict(width=1.5), fillcolor=shades[position % len(shades)],
            boxpoints="all", jitter=0.5, pointpos=0, marker=dict(size=4, opacity=0.55, line=dict(color="white", width=1)),
            showlegend=False, hovertemplate="VIX " + label + "<br>move %{y:+.2f}%<extra></extra>",
        )
    )
fig.add_hline(y=0, line_color=MUTED_INK, line_width=1)
fig.update_layout(
    title=f"{INDEX_NAME} — Every Expiry Cycle's Signed Move {HOLD_TITLE}, By India VIX At Entry",
    template="plotly_white", height=500, margin=dict(t=70, b=60, l=60, r=20),
)
fig.update_xaxes(title_text=f"India VIX at entry ({VIX_ANCHOR})", gridcolor=GRID)
fig.update_yaxes(title_text="Move over the window (%)", gridcolor=GRID, zeroline=False)
fig.show()

vix_rank_corr = rank_corr(cycles["vix"], cycles["abs_move_pct"])
level_corr = cycles["vix"].corr(cycles["abs_move_pct"])
print(f"VIX at entry vs |move|: Spearman {vix_rank_corr:+.3f}, Pearson {level_corr:+.3f} over {len(cycles)} cycles")
print(f"Overall: avg |move| {cycles['abs_move_pct'].mean():.2f}%, median {cycles['abs_move_pct'].median():.2f}%, "
      f"90th pct {cycles['abs_move_pct'].quantile(0.9):.2f}%, max {cycles['abs_move_pct'].max():.2f}%")
print(f"Avg signed move {cycles['move_pct'].mean():+.2f}%, closed up in "
      f"{(cycles['move_pct'] > 0).sum()} / {len(cycles)} cycles")

VIX at entry vs |move|: Spearman +0.199, Pearson +0.221 over 173 cycles
Overall: avg |move| 0.77%, median 0.60%, 90th pct 1.56%, max 3.95%
Avg signed move +0.07%, closed up in 92 / 173 cycles


## Touch Risk Inside The Window — What The Close-To-Close Move Does Not Show

The headline move is a settlement number: it ignores everything that happened in between. These two sessions carry
roughly fourteen hourly candles, and a short strangle can be stopped out by any one of them. For each bucket:

- **Avg max up / avg max down** — the best and worst hourly print inside the window, against the entry price.
- **Touched ±d** — the share of cycles where the index traded through that distance in *either* direction at some hour.
- **Settled beyond ±d** — the share where it was still outside that distance at the close of expiry.

The gap between the last two columns is what a seller keeps for sitting through the noise — and the share of cycles
where a stop would have fired on a strike that ultimately expired worthless.

In [9]:
touch_rows = []
for label, bucket in grouped.items():
    row = {"VIX band": label, "Cycles": len(bucket),
           "Avg max up %": bucket["max_up_pct"].mean(), "Avg max down %": bucket["max_down_pct"].mean()}
    for distance in TOUCH_DISTANCES:
        row[f"Touched ±{distance:g}%"] = (
            (bucket["max_up_pct"] >= distance) | (bucket["max_down_pct"] <= -distance)
        ).mean() * 100
        row[f"Settled ±{distance:g}%"] = (bucket["abs_move_pct"] >= distance).mean() * 100
    touch_rows.append(row)
touch = pd.DataFrame(touch_rows)

touch_cols = [f"Touched ±{d:g}%" for d in TOUCH_DISTANCES]
settle_cols = [f"Settled ±{d:g}%" for d in TOUCH_DISTANCES]
header = ["VIX<br>band", "Cycles", "Avg<br>max up", "Avg<br>max down"]
header += [f"Touched<br>±{d:g}%" for d in TOUCH_DISTANCES] + [f"Settled<br>±{d:g}%" for d in TOUCH_DISTANCES]
values = [touch["VIX band"], touch["Cycles"], touch["Avg max up %"].map("{:+.2f}%".format),
          touch["Avg max down %"].map("{:+.2f}%".format)]
values += [touch[col].map("{:.0f}%".format) for col in touch_cols + settle_cols]
fills = [["#f4f3f0"] * len(touch), ["white"] * len(touch),
         shade(touch["Avg max up %"], UP_COLOR), shade(touch["Avg max down %"], DOWN_COLOR)]
fills += [shade(touch[col], UP_COLOR) for col in touch_cols]
fills += [shade(touch[col], DOWN_COLOR) for col in settle_cols]

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[62, 52, 62, 68] + [62] * len(touch_cols) + [62] * len(settle_cols),
            header=dict(values=header, fill_color=HEADER_FILL, font=dict(color="white", size=10),
                        align="center", height=44),
            cells=dict(values=values, fill_color=fills, align="center", height=27, font=dict(size=11)),
        )
    ]
)
fig.update_layout(
    title=f"{INDEX_NAME} — Touched During The Window Vs Settled Beyond It, By India VIX At Entry",
    height=130 + 27 * len(touch) + 60, margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

## Does The Ordering Survive A Change Of Anchor Or Of Era?

Three things could make the main table an artefact rather than a finding, so each is checked against the same bucketing:

1. **The VIX anchor.** `entry_open` is the level quoted at the moment of entry; `prev_close` is the fully settled level
   from the session before. If the bucket averages move much between them, the table is reading an intraday VIX wobble
   rather than a regime.
2. **The era.** The sample is split at its median expiry date. Expiry weekday changed inside the sample (and for Sensex
   so did the weekly contract's age), so a monotone pattern that only exists in one half is not a tradable one.
3. **Where the move comes from.** The window is three legs — the entry session, the overnight gap, and expiry day
   itself. Their average magnitudes say whether the extra travel in a high-VIX bucket arrives while a position can be
   managed or while it cannot.

In [10]:
alt_anchor = "vix_prev_close" if VIX_ANCHOR == "entry_open" else "vix_entry_open"
alt_band = pd.cut(cycles[alt_anchor], VIX_BANDS, labels=BAND_LABELS)
midpoint = cycles["expiry"].median()
early, late_era = cycles[cycles["expiry"] <= midpoint], cycles[cycles["expiry"] > midpoint]

robust = pd.DataFrame(
    [
        {
            "VIX band": label,
            f"Avg |move| ({VIX_ANCHOR})": bucket["abs_move_pct"].mean(),
            "Cycles": len(bucket),
            f"Avg |move| ({alt_anchor.removeprefix('vix_')})": cycles.loc[alt_band == label, "abs_move_pct"].mean(),
            "Cycles (alt)": int((alt_band == label).sum()),
            f"Avg |move| to {midpoint:%Y-%m}": early.loc[early["band"] == label, "abs_move_pct"].mean(),
            f"Avg |move| after": late_era.loc[late_era["band"] == label, "abs_move_pct"].mean(),
            "Avg |entry day|": bucket["entry_day_pct"].abs().mean(),
            "Avg |gap|": bucket["gap_pct"].abs().mean(),
            "Avg |expiry day|": bucket["expiry_day_pct"].abs().mean(),
        }
        for label, bucket in grouped.items()
    ]
).set_index("VIX band")

display(robust.round(2))
print(f"Sample split at the median expiry {midpoint:%Y-%m-%d}: {len(early)} cycles before, {len(late_era)} after")
print(f"Spearman VIX vs |move| - {VIX_ANCHOR}: {rank_corr(cycles['vix'], cycles['abs_move_pct']):+.3f}, "
      f"{alt_anchor.removeprefix('vix_')}: {rank_corr(cycles[alt_anchor], cycles['abs_move_pct']):+.3f}")
print(f"Leg magnitudes over all cycles - entry day {cycles['entry_day_pct'].abs().mean():.2f}%, "
      f"overnight gap {cycles['gap_pct'].abs().mean():.2f}%, expiry day {cycles['expiry_day_pct'].abs().mean():.2f}%")
print(f"Expiry weekday counts: {cycles['expiry_day'].value_counts().to_dict()}")

,Avg |move| (entry_open),Cycles,Avg |move| (prev_close),Cycles (alt),Avg |move| to 2025-01,Avg |move| after,Avg |entry day|,Avg |gap|,Avg |expiry day|
VIX band,,,,,,,,,
≤ 10,0.40,2,0.45,3,NaN,0.40,0.36,0.05,0.13
10–11,0.61,14,0.67,10,0.80,0.43,0.36,0.25,0.38
11–12,0.58,41,0.59,42,0.62,0.53,0.44,0.28,0.43
12–13,0.75,24,0.73,25,0.87,0.67,0.47,0.27,0.50
13–14,0.96,26,0.88,28,0.76,1.21,0.49,0.37,0.53
14–15,0.70,24,0.70,22,0.69,0.73,0.60,0.16,0.46
15–16,0.66,12,0.58,11,0.88,0.37,0.45,0.34,0.57
16–18,0.99,14,1.07,16,1.06,0.95,0.56,0.42,0.57
> 18,1.14,16,1.14,16,1.19,1.12,0.62,0.66,0.69


Sample split at the median expiry 2025-01-21: 87 cycles before, 86 after
Spearman VIX vs |move| - entry_open: +0.199, prev_close: +0.209
Leg magnitudes over all cycles - entry day 0.49%, overnight gap 0.32%, expiry day 0.50%
Expiry weekday counts: {'Friday': 79, 'Thursday': 55, 'Tuesday': 34, 'Wednesday': 5}


## What The Numbers Say

Everything in this section is computed from the cells above; nothing is typed in by hand.

In [11]:
lowest, highest = bands.iloc[0], bands.iloc[-1]
richest = bands.loc[bands["Actual / implied"].idxmin()]
by_year = cycles.groupby("year")["abs_move_pct"].agg(["mean", "size"])

display(Markdown(f"""
- Over **{len(cycles)} expiry cycles** ({cycles['expiry'].min():%b %Y} to {cycles['expiry'].max():%b %Y}),
  {INDEX_NAME} moved **{cycles['abs_move_pct'].mean():.2f}%** on average from the {HOLD_LABEL}, median
  **{cycles['abs_move_pct'].median():.2f}%**. The signed average is **{cycles['move_pct'].mean():+.2f}%** — the window
  carries travel, not direction.
- The lowest VIX bucket (**{lowest['VIX band']}**, n={lowest['Cycles']:.0f}) averaged
  **{lowest['Avg |move| %']:.2f}%**; the highest (**{highest['VIX band']}**, n={highest['Cycles']:.0f}) averaged
  **{highest['Avg |move| %']:.2f}%** — a factor of
  **{highest['Avg |move| %'] / lowest['Avg |move| %']:.1f}x** across the VIX range.
- Rank correlation between VIX at entry and the size of the move that followed:
  **{vix_rank_corr:+.3f}** (Spearman, per cycle). VIX orders the *scale* of these two sessions
  {'well' if vix_rank_corr > 0.35 else 'only loosely'}, which is as much as a vol quote claims to do.
- Against the quote: realised |move| was **{cycles['abs_move_pct'].mean() / cycles['implied_abs_move_pct'].mean():.2f}x**
  the VIX-implied expected absolute move overall. The bucket where the quote was richest relative to what followed is
  **{richest['VIX band']}** at **{richest['Actual / implied']:.2f}x** (realised {richest['Avg |move| %']:.2f}% vs implied
  {richest['Implied E|move| %']:.2f}%).
- Inside the window the index traded **{cycles['max_up_pct'].mean():+.2f}%** up and
  **{cycles['max_down_pct'].mean():+.2f}%** down on average, against a settled move of
  {cycles['abs_move_pct'].mean():.2f}% — the gap between the touch columns and the settle columns above.
- Of the three legs, expiry day itself contributes **{cycles['expiry_day_pct'].abs().mean():.2f}%**, the entry session
  **{cycles['entry_day_pct'].abs().mean():.2f}%**, and the unmanageable overnight gap
  **{cycles['gap_pct'].abs().mean():.2f}%**.
- Per year: {', '.join(f"{int(year)} {row['mean']:.2f}% (n={int(row['size'])})" for year, row in by_year.iterrows())}.
"""))


- Over **173 expiry cycles** (May 2023 to Sep 2026),
  SENSEX moved **0.77%** on average from the open on expiry_day-1 to close on expiry_day, median
  **0.60%**. The signed average is **+0.07%** — the window
  carries travel, not direction.
- The lowest VIX bucket (**≤ 10**, n=2) averaged
  **0.40%**; the highest (**> 18**, n=16) averaged
  **1.14%** — a factor of
  **2.9x** across the VIX range.
- Rank correlation between VIX at entry and the size of the move that followed:
  **+0.199** (Spearman, per cycle). VIX orders the *scale* of these two sessions
  only loosely, which is as much as a vol quote claims to do.
- Against the quote: realised |move| was **0.78x**
  the VIX-implied expected absolute move overall. The bucket where the quote was richest relative to what followed is
  **≤ 10** at **0.58x** (realised 0.40% vs implied
  0.69%).
- Inside the window the index traded **+0.78%** up and
  **-0.74%** down on average, against a settled move of
  0.77% — the gap between the touch columns and the settle columns above.
- Of the three legs, expiry day itself contributes **0.50%**, the entry session
  **0.49%**, and the unmanageable overnight gap
  **0.32%**.
- Per year: 2023 0.66% (n=32), 2024 0.83% (n=52), 2025 0.76% (n=51), 2026 0.79% (n=38).
